# Fase 4 — Modelagem

**Projeto**: Mineração de Padrões Frequentes em Acidentes de Rodovias Federais  
**Grupo 18**: Erik Roberto Reis Neves, Gabriel Campos Prudente, Felipe Silva Fraga Damasceno  
**Disciplina**: Mineração de Dados — UFMG  

---

## Objetivos desta fase
1. Aplicar FP-Growth global
2. Extrair regras de associação filtradas e com métricas de interesse (Lift > 1)
3. Focar em regras relacionadas à severidade e fatalidade dos acidentes
4. Explorar variações paramétricas (Grid Search)
5. Criar e avaliar recortes locais (via Clustering de características descritivas) e Janelas temporais

**Alinhamento CRISP-DM**: *Modeling*


## Setup — Importações e Configurações

In [1]:
"""
Fase 4 -- Modelagem (script executavel)
FP-Growth para mineracao de padroes frequentes e regras de associacao.
"""
import pandas as pd
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
from mlxtend.frequent_patterns import fpgrowth, association_rules
from sklearn.cluster import KMeans
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.metrics import silhouette_score
import time
import json
import warnings
import pickle
warnings.filterwarnings('ignore')

## Configuracoes

In [2]:
plt.rcParams['figure.figsize'] = (14, 6)
plt.rcParams['font.size'] = 12
plt.rcParams['axes.titlesize'] = 14
plt.rcParams['axes.labelsize'] = 12
plt.rcParams['figure.dpi'] = 100
sns.set_style('whitegrid')

pd.set_option('display.max_columns', 50)
pd.set_option('display.max_rows', 100)
pd.set_option('display.float_format', '{:.4f}'.format)

PROCESSED_DIR = Path('../data/processed/')
OUTPUT_DIR = Path('../outputs/')
FIGURAS_DIR = OUTPUT_DIR / 'figuras'
TABELAS_DIR = OUTPUT_DIR / 'tabelas'
DADOS_DIR = OUTPUT_DIR / 'dados'
for d in [FIGURAS_DIR, TABELAS_DIR, DADOS_DIR]:
    d.mkdir(parents=True, exist_ok=True)

print('[OK] Imports e configuracoes realizados.')

[OK] Imports e configuracoes realizados.


## 4.0 Carregamento Dos Dados Processados

In [3]:
print('\n' + '='*60)
print('4.0 CARREGAMENTO DOS DADOS PROCESSADOS')
print('='*60)

df_onehot = pd.read_pickle(PROCESSED_DIR / 'transacional.pkl')
df = pd.read_pickle(PROCESSED_DIR / 'df_limpo.pkl')

with open(PROCESSED_DIR / 'preparacao_metadata.json', 'r', encoding='utf-8') as f:
    metadata = json.load(f)

print(f'DataFrame limpo: {len(df):,} registros x {df.shape[1]} colunas')
print(f'Transacional (filtrado): {df_onehot.shape[0]:,} transacoes x {df_onehot.shape[1]} itens')
print(f'Dataset: {metadata["dataset_ativo"]} (UF={metadata["filtro_uf"]})')


4.0 CARREGAMENTO DOS DADOS PROCESSADOS
DataFrame limpo: 2,985 registros x 35 colunas
Transacional (filtrado): 2,985 transacoes x 83 itens
Dataset: por_ocorrencia (UF=MG)


## 4.1 Mineracao Global — FP-Growth

In [4]:
print('\n' + '='*60)
print('4.1 MINERACAO GLOBAL -- FP-GROWTH')
print('='*60)

min_support = 0.05
print(f'Suporte minimo: {min_support*100:.0f}%')

t0 = time.time()
itemsets = fpgrowth(df_onehot, min_support=min_support, use_colnames=True)
t_fpgrowth = time.time() - t0

# Adicionar coluna com tamanho do itemset
itemsets['length'] = itemsets['itemsets'].apply(len)

print(f'\n[OK] FP-Growth concluido em {t_fpgrowth:.2f}s')
print(f'Itemsets frequentes encontrados: {len(itemsets)}')
print(f'\nDistribuicao por tamanho:')
print(itemsets['length'].value_counts().sort_index())

# Top 20 itemsets por suporte
print(f'\nTop 20 itemsets por suporte:')
top_itemsets = itemsets.nlargest(20, 'support')
for _, row in top_itemsets.iterrows():
    items_str = ', '.join(sorted([str(i) for i in row['itemsets']]))
    print(f'  [{row["length"]}] sup={row["support"]:.3f} | {items_str}')


4.1 MINERACAO GLOBAL -- FP-GROWTH
Suporte minimo: 5%

[OK] FP-Growth concluido em 0.18s
Itemsets frequentes encontrados: 2467

Distribuicao por tamanho:


length
1     43
2    304
3    754
4    829
5    435
6     98
7      4
Name: count, dtype: int64

Top 20 itemsets por suporte:


  [1] sup=0.929 | gravidade_binaria_Não Fatal
  [1] sup=0.793 | classificacao_acidente_Com Vítimas Feridas
  [2] sup=0.793 | classificacao_acidente_Com Vítimas Feridas, gravidade_binaria_Não Fatal
  [1] sup=0.705 | uso_solo_Não
  [1] sup=0.682 | fim_de_semana_Não
  [2] sup=0.651 | gravidade_binaria_Não Fatal, uso_solo_Não
  [2] sup=0.636 | fim_de_semana_Não, gravidade_binaria_Não Fatal
  [1] sup=0.582 | fase_dia_Pleno dia
  [2] sup=0.549 | classificacao_acidente_Com Vítimas Feridas, uso_solo_Não
  [3] sup=0.549 | classificacao_acidente_Com Vítimas Feridas, gravidade_binaria_Não Fatal, uso_solo_Não
  [2] sup=0.549 | classificacao_acidente_Com Vítimas Feridas, fim_de_semana_Não
  [3] sup=0.549 | classificacao_acidente_Com Vítimas Feridas, fim_de_semana_Não, gravidade_binaria_Não Fatal
  [2] sup=0.549 | fase_dia_Pleno dia, gravidade_binaria_Não Fatal
  [1] sup=0.514 | condicao_metereologica_Céu Claro
  [1] sup=0.486 | tipo_pista_Simples
  [2] sup=0.480 | classificacao_acidente_Com Vítimas

## 4.2 Geracao De Regras De Associacao

In [5]:
print('\n' + '='*60)
print('4.2 GERACAO DE REGRAS DE ASSOCIACAO')
print('='*60)

t0 = time.time()
rules = association_rules(itemsets, metric='confidence', min_threshold=0.5, num_itemsets=len(df_onehot))
t_rules = time.time() - t0

# Filtrar regras com lift > 1 (associacoes positivas)
rules = rules[rules['lift'] > 1.0].copy()

# Ordenar por lift
rules = rules.sort_values('lift', ascending=False)

print(f'\n[OK] Regras geradas em {t_rules:.2f}s')
print(f'Regras com lift > 1: {len(rules)}')

if len(rules) > 0:
    print(f'\nEstatisticas das regras:')
    for col in ['support', 'confidence', 'lift', 'leverage', 'conviction']:
        if col in rules.columns:
            print(f'  {col}: min={rules[col].min():.3f}, med={rules[col].median():.3f}, max={rules[col].max():.3f}')
    
    # Adicionar colunas de texto para facilitar analise
    rules['antecedents_str'] = rules['antecedents'].apply(lambda x: ' & '.join(sorted([str(i) for i in x])))
    rules['consequents_str'] = rules['consequents'].apply(lambda x: ' & '.join(sorted([str(i) for i in x])))
    rules['n_antecedents'] = rules['antecedents'].apply(len)
    rules['n_consequents'] = rules['consequents'].apply(len)
    
    # Top 30 regras por lift
    print(f'\nTop 30 regras por lift:')
    for i, (_, row) in enumerate(rules.head(30).iterrows()):
        print(f'  {i+1:2d}. [{row["antecedents_str"]}] => [{row["consequents_str"]}]')
        print(f'      sup={row["support"]:.3f} conf={row["confidence"]:.3f} lift={row["lift"]:.2f}')


4.2 GERACAO DE REGRAS DE ASSOCIACAO

[OK] Regras geradas em 0.12s
Regras com lift > 1: 7923

Estatisticas das regras:
  support: min=0.050, med=0.071, max=0.793
  confidence: min=0.500, med=0.721, max=1.000
  lift: min=1.000, med=1.104, max=14.147
  leverage: min=0.000, med=0.008, max=0.151
  conviction: min=1.000, med=1.360, max=inf

Top 30 regras por lift:
   1. [classificacao_acidente_Com Vítimas Fatais] => [gravidade_binaria_Fatal & uso_solo_Não]
      sup=0.054 conf=0.768 lift=14.15
   2. [gravidade_binaria_Fatal] => [classificacao_acidente_Com Vítimas Fatais & uso_solo_Não]
      sup=0.054 conf=0.768 lift=14.15
   3. [gravidade_binaria_Fatal] => [classificacao_acidente_Com Vítimas Fatais]
      sup=0.071 conf=1.000 lift=14.15
   4. [classificacao_acidente_Com Vítimas Fatais & uso_solo_Não] => [gravidade_binaria_Fatal]
      sup=0.054 conf=1.000 lift=14.15
   5. [gravidade_binaria_Fatal & uso_solo_Não] => [classificacao_acidente_Com Vítimas Fatais]
      sup=0.054 conf=1.000 lift

## 4.3 Foco Em Regras De Gravidade

In [6]:
print('\n' + '='*60)
print('4.3 FOCO EM REGRAS DE GRAVIDADE')
print('='*60)

if len(rules) > 0:
    # Regras que predizem acidentes fatais (consequente contem classificacao Fatal)
    rules_fatais = rules[rules['consequents'].apply(
        lambda x: any('Fatal' in str(i) and 'Fatais' in str(i) for i in x)
    )].copy()
    
    # Regras que predizem gravidade binaria fatal
    rules_grav_fatal = rules[rules['consequents'].apply(
        lambda x: any('gravidade_binaria=Fatal' in str(i) for i in x)
    )].copy()
    
    # Combinar (uniao) - regras que predizem qualquer indicador de fatalidade
    idx_fatal = set(rules_fatais.index).union(set(rules_grav_fatal.index))
    rules_graves = rules.loc[list(idx_fatal)].sort_values('lift', ascending=False).copy()
    
    # Regras sobre tipo de acidente no consequente
    rules_tipo = rules[rules['consequents'].apply(
        lambda x: any('tipo_acidente=' in str(i) for i in x)
    )].copy()
    
    # Regras sobre causa do acidente no consequente
    rules_causa = rules[rules['consequents'].apply(
        lambda x: any('causa_acidente=' in str(i) for i in x)
    )].copy()
    
    print(f'Regras com consequente "acidentes fatais": {len(rules_fatais)}')
    print(f'Regras com consequente "gravidade_binaria=Fatal": {len(rules_grav_fatal)}')
    print(f'Regras graves (uniao): {len(rules_graves)}')
    print(f'Regras sobre tipo de acidente: {len(rules_tipo)}')
    print(f'Regras sobre causa do acidente: {len(rules_causa)}')
    
    if len(rules_graves) > 0:
        print(f'\nTop 20 regras que predizem acidentes fatais/graves (por lift):')
        for i, (_, row) in enumerate(rules_graves.head(20).iterrows()):
            print(f'  {i+1:2d}. [{row["antecedents_str"]}] => [{row["consequents_str"]}]')
            print(f'      sup={row["support"]:.3f} conf={row["confidence"]:.3f} lift={row["lift"]:.2f}')
    else:
        print('\n[!] Nenhuma regra de gravidade encontrada com os parametros atuais.')
        print('    Considere diminuir min_support ou min_confidence.')


4.3 FOCO EM REGRAS DE GRAVIDADE


Regras com consequente "acidentes fatais": 0
Regras com consequente "gravidade_binaria=Fatal": 0
Regras graves (uniao): 0
Regras sobre tipo de acidente: 0
Regras sobre causa do acidente: 0

[!] Nenhuma regra de gravidade encontrada com os parametros atuais.
    Considere diminuir min_support ou min_confidence.


## 4.4 Experimentacao Com Parametros

In [7]:
print('\n' + '='*60)
print('4.4 EXPERIMENTACAO COM PARAMETROS (GRID SEARCH)')
print('='*60)

support_values = [0.01, 0.03, 0.05, 0.10]
confidence_values = [0.3, 0.5, 0.7]
lift_values = [1.0, 1.5, 2.0]

resultados_grid = []

for ms in support_values:
    t0 = time.time()
    try:
        its = fpgrowth(df_onehot, min_support=ms, use_colnames=True)
    except Exception as e:
        print(f'  [!] Erro com min_support={ms}: {e}')
        continue
    t_fp = time.time() - t0
    
    if len(its) == 0:
        print(f'  min_support={ms}: 0 itemsets')
        continue
    
    for mc in confidence_values:
        t0 = time.time()
        try:
            rls = association_rules(its, metric='confidence', min_threshold=mc, num_itemsets=len(df_onehot))
        except Exception as e:
            print(f'  [!] Erro com min_conf={mc}: {e}')
            continue
        t_rl = time.time() - t0
        
        for ml in lift_values:
            rls_f = rls[rls['lift'] > ml]
            
            # Contar regras de gravidade
            n_grav = len(rls_f[rls_f['consequents'].apply(
                lambda x: any('Fatal' in str(i) for i in x)
            )])
            
            resultado = {
                'min_support': ms,
                'min_confidence': mc,
                'min_lift': ml,
                'n_itemsets': len(its),
                'n_regras': len(rls_f),
                'n_regras_gravidade': n_grav,
                'tempo_fpgrowth_s': round(t_fp, 3),
                'tempo_regras_s': round(t_rl, 3),
            }
            resultados_grid.append(resultado)

df_grid = pd.DataFrame(resultados_grid)
print(f'\nResultados do Grid Search ({len(df_grid)} combinacoes):')
print(df_grid.to_string(index=False))

# Salvar grid
df_grid.to_csv(DADOS_DIR / 'grid_parametros.csv', index=False)
print(f'\n[OK] Grid salvo em {DADOS_DIR / "grid_parametros.csv"}')

# Visualizacao do grid
fig, axes = plt.subplots(1, 3, figsize=(18, 6))

# Painel 1: N itemsets vs support
for mc in confidence_values:
    mask = (df_grid['min_confidence'] == mc) & (df_grid['min_lift'] == 1.0)
    subset = df_grid[mask]
    if len(subset) > 0:
        axes[0].plot(subset['min_support'], subset['n_regras'], 'o-', label=f'conf>={mc}')
axes[0].set_xlabel('Suporte Minimo')
axes[0].set_ylabel('Numero de Regras')
axes[0].set_title('Regras vs. Suporte Minimo (lift>1.0)')
axes[0].legend()
axes[0].set_yscale('log')

# Painel 2: N regras vs lift
for ms in support_values:
    mask = (df_grid['min_support'] == ms) & (df_grid['min_confidence'] == 0.5)
    subset = df_grid[mask]
    if len(subset) > 0:
        axes[1].plot(subset['min_lift'], subset['n_regras'], 'o-', label=f'sup>={ms}')
axes[1].set_xlabel('Lift Minimo')
axes[1].set_ylabel('Numero de Regras')
axes[1].set_title('Regras vs. Lift Minimo (conf>=0.5)')
axes[1].legend()

# Painel 3: Regras de gravidade
for ms in support_values:
    mask = (df_grid['min_support'] == ms) & (df_grid['min_confidence'] == 0.5)
    subset = df_grid[mask]
    if len(subset) > 0:
        axes[2].plot(subset['min_lift'], subset['n_regras_gravidade'], 's-', label=f'sup>={ms}')
axes[2].set_xlabel('Lift Minimo')
axes[2].set_ylabel('Regras de Gravidade')
axes[2].set_title('Regras de Gravidade vs. Lift (conf>=0.5)')
axes[2].legend()

plt.tight_layout()
plt.savefig(FIGURAS_DIR / 'grid_parametros.png', dpi=300, bbox_inches='tight')
plt.close()
print(f'[OK] Visualizacao do grid salva em {FIGURAS_DIR / "grid_parametros.png"}')


4.4 EXPERIMENTACAO COM PARAMETROS (GRID SEARCH)



Resultados do Grid Search (36 combinacoes):
 min_support  min_confidence  min_lift  n_itemsets  n_regras  n_regras_gravidade  tempo_fpgrowth_s  tempo_regras_s
      0.0100          0.3000    1.0000       31933    275243               98375            1.0290          5.4010
      0.0100          0.3000    1.5000       31933    121567               40509            1.0290          5.4010
      0.0100          0.3000    2.0000       31933     68817               22524            1.0290          5.4010
      0.0100          0.5000    1.0000       31933    150382               56354            1.0290          4.2910
      0.0100          0.5000    1.5000       31933     60135               19820            1.0290          4.2910
      0.0100          0.5000    2.0000       31933     32443               10461            1.0290          4.2910
      0.0100          0.7000    1.0000       31933     79296               33003            1.0290          3.4910
      0.0100          0.7000    1.5

[OK] Visualizacao do grid salva em ..\outputs\figuras\grid_parametros.png


## 4.5 Clustering + Mineracao Local

In [8]:
print('\n' + '='*60)
print('4.5 CLUSTERING + MINERACAO LOCAL')
print('='*60)

# Preparacao para clustering - usar colunas categoricas do df limpo
COLUNAS_CATEGORICAS = [
    'dia_semana', 'causa_acidente', 'tipo_acidente',
    'classificacao_acidente', 'fase_dia', 'sentido_via',
    'condicao_metereologica', 'tipo_pista', 'tracado_via', 'uso_solo',
]

# Verificar quais colunas estao disponiveis
cols_disponiveis = [c for c in COLUNAS_CATEGORICAS if c in df.columns]

df_cluster = df[cols_disponiveis].copy()
le_dict = {}
for col in df_cluster.columns:
    le = LabelEncoder()
    df_cluster[col] = le.fit_transform(df_cluster[col].astype(str))
    le_dict[col] = le

scaler = StandardScaler()
X_scaled = scaler.fit_transform(df_cluster)

print(f'Dados para clustering: {X_scaled.shape[0]} amostras x {X_scaled.shape[1]} features')

# 4.5.2 Determinacao do numero de clusters (Cotovelo + Silhouette)
K_range = range(2, 10)
inertias = []
silhouettes = []

for k in K_range:
    km_model = KMeans(n_clusters=k, random_state=42, n_init=10)
    labels = km_model.fit_predict(X_scaled)
    inertias.append(km_model.inertia_)
    sil = silhouette_score(X_scaled, labels, sample_size=min(5000, len(X_scaled)))
    silhouettes.append(sil)
    print(f'  k={k}: inertia={km_model.inertia_:.1f}, silhouette={sil:.3f}')

# Plotar cotovelo + silhouette
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].plot(list(K_range), inertias, 'bo-')
axes[0].set_xlabel('Numero de Clusters (k)')
axes[0].set_ylabel('Inercia')
axes[0].set_title('Metodo do Cotovelo')

axes[1].plot(list(K_range), silhouettes, 'rs-')
axes[1].set_xlabel('Numero de Clusters (k)')
axes[1].set_ylabel('Silhouette Score')
axes[1].set_title('Silhouette Score')

plt.tight_layout()
plt.savefig(FIGURAS_DIR / 'clustering_cotovelo_silhouette.png', dpi=300, bbox_inches='tight')
plt.close()
print(f'\n[OK] Graficos de cotovelo/silhouette salvos')

# Escolher k com melhor silhouette
best_k = list(K_range)[np.argmax(silhouettes)]
print(f'\nMelhor k pelo Silhouette Score: k={best_k} (score={max(silhouettes):.3f})')

# Aplicar clustering com k escolhido
k_final = best_k
km_final = KMeans(n_clusters=k_final, random_state=42, n_init=10)
df['cluster'] = km_final.fit_predict(X_scaled)

print(f'\nDistribuicao dos clusters:')
print(df['cluster'].value_counts().sort_index())

# Perfil dos clusters
print(f'\nPerfil dos clusters (top 3 valores por coluna):')
for c in range(k_final):
    mask = df['cluster'] == c
    n = mask.sum()
    print(f'\n--- Cluster {c} ({n} registros, {n/len(df)*100:.1f}%) ---')
    for col in cols_disponiveis[:5]:  # Top 5 colunas
        top_val = df.loc[mask, col].value_counts().head(2)
        vals = ', '.join([f'{v}({cnt})' for v, cnt in top_val.items()])
        print(f'  {col}: {vals}')

# 4.5.3 Mineracao por cluster
print(f'\n=== Mineracao de regras por cluster ===')
rules_por_cluster = {}
for c in range(k_final):
    mask = df['cluster'] == c
    # Usar os mesmos indices para selecionar transacoes do one-hot
    idx = df.index[mask]
    common_idx = df_onehot.index.intersection(idx)
    df_c = df_onehot.loc[common_idx]
    
    if len(df_c) < 50:
        print(f'  Cluster {c}: apenas {len(df_c)} registros, pulando')
        continue
    
    try:
        itemsets_c = fpgrowth(df_c, min_support=0.05, use_colnames=True)
        if len(itemsets_c) > 0:
            rules_c = association_rules(itemsets_c, metric='confidence', min_threshold=0.5, num_itemsets=len(df_c))
            rules_c = rules_c[rules_c['lift'] > 1.0].copy()
            rules_c['antecedents_str'] = rules_c['antecedents'].apply(lambda x: ' & '.join(sorted([str(i) for i in x])))
            rules_c['consequents_str'] = rules_c['consequents'].apply(lambda x: ' & '.join(sorted([str(i) for i in x])))
            rules_por_cluster[c] = rules_c
            print(f'  Cluster {c}: {mask.sum()} registros, {len(itemsets_c)} itemsets, {len(rules_c)} regras')
        else:
            print(f'  Cluster {c}: {mask.sum()} registros, 0 itemsets')
    except Exception as e:
        print(f'  Cluster {c}: erro - {e}')

# Top regras por cluster
for c, rules_c in rules_por_cluster.items():
    if len(rules_c) > 0:
        print(f'\n  Top 5 regras do Cluster {c} (por lift):')
        for i, (_, row) in enumerate(rules_c.nlargest(5, 'lift').iterrows()):
            print(f'    {i+1}. [{row["antecedents_str"]}] => [{row["consequents_str"]}]')
            print(f'       sup={row["support"]:.3f} conf={row["confidence"]:.3f} lift={row["lift"]:.2f}')


4.5 CLUSTERING + MINERACAO LOCAL
Dados para clustering: 2985 amostras x 10 features


  k=2: inertia=26648.0, silhouette=0.120


  k=3: inertia=24496.5, silhouette=0.118


  k=4: inertia=23029.7, silhouette=0.109


  k=5: inertia=21681.8, silhouette=0.115


...

  k=8: inertia=19366.5, silhouette=0.096


  k=9: inertia=18827.4, silhouette=0.105



[OK] Graficos de cotovelo/silhouette salvos

Melhor k pelo Silhouette Score: k=2 (score=0.120)

Distribuicao dos clusters:
cluster
0     880
1    2105
Name: count, dtype: int64

Perfil dos clusters (top 3 valores por coluna):

--- Cluster 0 (880 registros, 29.5%) ---
  dia_semana: sábado(136), sexta-feira(132)
  causa_acidente: Ausência de reação do condutor(151), Reação tardia ou ineficiente do condutor(118)
  tipo_acidente: Colisão traseira(214), Colisão transversal(155)
  classificacao_acidente: Com Vítimas Feridas(730), Sem Vítimas(101)
  fase_dia: Pleno dia(504), Plena Noite(300)

--- Cluster 1 (2105 registros, 70.5%) ---
  dia_semana: domingo(372), sábado(319)
  causa_acidente: Ausência de reação do condutor(354), Velocidade Incompatível(306)
  tipo_acidente: Saída de leito carroçável(559), Colisão traseira(304)
  classificacao_acidente: Com Vítimas Feridas(1638), Sem Vítimas(305)
  fase_dia: Pleno dia(1232), Plena Noite(674)

=== Mineracao de regras por cluster ===


  Cluster 0: 880 registros, 3517 itemsets, 19824 regras


  Cluster 1: 2105 registros, 2975 itemsets, 11859 regras

  Top 5 regras do Cluster 0 (por lift):
    1. [gravidade_binaria_Fatal] => [classificacao_acidente_Com Vítimas Fatais]
       sup=0.056 conf=1.000 lift=17.96
    2. [classificacao_acidente_Com Vítimas Fatais] => [gravidade_binaria_Fatal]
       sup=0.056 conf=1.000 lift=17.96
    3. [gravidade_binaria_Fatal & uso_solo_Sim] => [classificacao_acidente_Com Vítimas Fatais]
       sup=0.056 conf=1.000 lift=17.96
    4. [classificacao_acidente_Com Vítimas Fatais & uso_solo_Sim] => [gravidade_binaria_Fatal]
       sup=0.056 conf=1.000 lift=17.96
    5. [gravidade_binaria_Fatal] => [classificacao_acidente_Com Vítimas Fatais & uso_solo_Sim]
       sup=0.056 conf=1.000 lift=17.96

  Top 5 regras do Cluster 1 (por lift):
    1. [gravidade_binaria_Fatal] => [classificacao_acidente_Com Vítimas Fatais & fim_de_semana_Não]
       sup=0.051 conf=0.660 lift=12.99
    2. [classificacao_acidente_Com Vítimas Fatais] => [fim_de_semana_Não & gravida

## 4.6 Analise Temporal (Mineracao Por Periodo)

In [9]:
print('\n' + '='*60)
print('4.6 ANALISE TEMPORAL (MINERACAO POR PERIODO)')
print('='*60)

# Usar a coluna 'mes' ja criada na fase 3
if 'mes' in df.columns:
    meses_unicos = sorted(df['mes'].unique())
    print(f'Meses disponiveis: {meses_unicos}')
    
    rules_por_periodo = {}
    for mes in meses_unicos:
        mask = df['mes'] == mes
        idx = df.index[mask]
        common_idx = df_onehot.index.intersection(idx)
        df_m = df_onehot.loc[common_idx]
        
        if len(df_m) < 100:
            print(f'  {mes}: apenas {len(df_m)} registros, pulando')
            continue
        
        try:
            itemsets_m = fpgrowth(df_m, min_support=0.05, use_colnames=True)
            if len(itemsets_m) > 0:
                rules_m = association_rules(itemsets_m, metric='confidence', min_threshold=0.5, num_itemsets=len(df_m))
                rules_m = rules_m[rules_m['lift'] > 1.0].copy()
                rules_m['antecedents_str'] = rules_m['antecedents'].apply(lambda x: ' & '.join(sorted([str(i) for i in x])))
                rules_m['consequents_str'] = rules_m['consequents'].apply(lambda x: ' & '.join(sorted([str(i) for i in x])))
                rules_por_periodo[mes] = rules_m
                
                # Contar regras de gravidade
                n_grav = len(rules_m[rules_m['consequents'].apply(
                    lambda x: any('Fatal' in str(i) for i in x)
                )])
                
                print(f'  {mes}: {mask.sum()} registros, {len(itemsets_m)} itemsets, {len(rules_m)} regras ({n_grav} de gravidade)')
            else:
                print(f'  {mes}: {mask.sum()} registros, 0 itemsets')
        except Exception as e:
            print(f'  {mes}: erro - {e}')
    
    # Comparar estabilidade temporal
    if len(rules_por_periodo) >= 2:
        print(f'\n=== Estabilidade temporal das regras ===')
        
        # Criar chave para cada regra
        from collections import Counter
        
        def regra_key(row):
            return (frozenset(row['antecedents']), frozenset(row['consequents']))
        
        contagem = Counter()
        for periodo, rules_p in rules_por_periodo.items():
            for _, row in rules_p.iterrows():
                contagem[regra_key(row)] += 1
        
        n_periodos = len(rules_por_periodo)
        regras_estaveis = {k for k, v in contagem.items() if v >= n_periodos * 0.5}
        regras_transitorias = {k for k, v in contagem.items() if v < n_periodos * 0.3}
        
        print(f'Total de regras unicas: {len(contagem)}')
        print(f'Regras estaveis (>= {n_periodos*0.5:.0f} periodos): {len(regras_estaveis)}')
        print(f'Regras transitorias (< {n_periodos*0.3:.0f} periodos): {len(regras_transitorias)}')
        
        # Top regras estaveis
        print(f'\nTop 10 regras mais estaveis (aparecem em mais periodos):')
        top_estaveis = contagem.most_common(10)
        for i, (key, cnt) in enumerate(top_estaveis):
            ante = ' & '.join(sorted([str(i) for i in key[0]]))
            cons = ' & '.join(sorted([str(i) for i in key[1]]))
            print(f'  {i+1:2d}. [{ante}] => [{cons}] (aparece em {cnt}/{n_periodos} periodos)')


4.6 ANALISE TEMPORAL (MINERACAO POR PERIODO)
Meses disponiveis: ['Mes_01', 'Mes_02', 'Mes_03', 'Mes_04']


  Mes_01: 797 registros, 2788 itemsets, 10769 regras (4125 de gravidade)


  Mes_02: 665 registros, 2466 itemsets, 7980 regras (3156 de gravidade)


  Mes_03: 807 registros, 2510 itemsets, 8393 regras (3306 de gravidade)


  Mes_04: 716 registros, 2823 itemsets, 11476 regras (4352 de gravidade)

=== Estabilidade temporal das regras ===


Total de regras unicas: 21726
Regras estaveis (>= 2 periodos): 9021
Regras transitorias (< 1 periodos): 12705

Top 10 regras mais estaveis (aparecem em mais periodos):
   1. [gravidade_binaria_Não Fatal] => [classificacao_acidente_Com Vítimas Feridas] (aparece em 4/4 periodos)
   2. [classificacao_acidente_Com Vítimas Feridas] => [gravidade_binaria_Não Fatal] (aparece em 4/4 periodos)
   3. [classificacao_acidente_Com Vítimas Feridas & uso_solo_Não] => [gravidade_binaria_Não Fatal] (aparece em 4/4 periodos)
   4. [gravidade_binaria_Não Fatal & uso_solo_Não] => [classificacao_acidente_Com Vítimas Feridas] (aparece em 4/4 periodos)
   5. [classificacao_acidente_Com Vítimas Feridas] => [gravidade_binaria_Não Fatal & uso_solo_Não] (aparece em 4/4 periodos)
   6. [gravidade_binaria_Não Fatal] => [classificacao_acidente_Com Vítimas Feridas & uso_solo_Não] (aparece em 4/4 periodos)
   7. [fim_de_semana_Não] => [classificacao_acidente_Com Vítimas Feridas] (aparece em 4/4 periodos)
   8. [class

## Salvamento De Resultados

In [10]:
print('\n' + '='*60)
print('SALVAMENTO DE RESULTADOS')
print('='*60)

# Salvar regras globais
if len(rules) > 0:
    cols_salvar = ['antecedents_str', 'consequents_str', 'support', 'confidence', 'lift', 'leverage', 'conviction', 'n_antecedents', 'n_consequents']
    cols_disp = [c for c in cols_salvar if c in rules.columns]
    rules[cols_disp].to_csv(DADOS_DIR / 'regras_completas.csv', index=False, encoding='utf-8-sig')
    print(f'[OK] Regras completas: {DADOS_DIR / "regras_completas.csv"} ({len(rules)} regras)')

# Salvar itemsets
itemsets_salvar = itemsets.copy()
itemsets_salvar['itemsets_str'] = itemsets_salvar['itemsets'].apply(lambda x: ', '.join(sorted([str(i) for i in x])))
itemsets_salvar[['itemsets_str', 'support', 'length']].to_csv(DADOS_DIR / 'itemsets_frequentes.csv', index=False, encoding='utf-8-sig')
print(f'[OK] Itemsets frequentes: {DADOS_DIR / "itemsets_frequentes.csv"} ({len(itemsets)} itemsets)')

# Salvar regras de gravidade
if len(rules_graves) > 0:
    cols_disp = [c for c in cols_salvar if c in rules_graves.columns]
    rules_graves[cols_disp].to_csv(DADOS_DIR / 'regras_gravidade.csv', index=False, encoding='utf-8-sig')
    print(f'[OK] Regras de gravidade: {DADOS_DIR / "regras_gravidade.csv"} ({len(rules_graves)} regras)')

# Salvar regras por periodo (pickle)
if len(rules_por_periodo) > 0:
    with open(PROCESSED_DIR / 'rules_por_periodo.pkl', 'wb') as f:
        pickle.dump(rules_por_periodo, f)
    print(f'[OK] Regras por periodo: {PROCESSED_DIR / "rules_por_periodo.pkl"} ({len(rules_por_periodo)} periodos)')

# Salvar regras por cluster (pickle)
if len(rules_por_cluster) > 0:
    with open(PROCESSED_DIR / 'rules_por_cluster.pkl', 'wb') as f:
        pickle.dump(rules_por_cluster, f)
    print(f'[OK] Regras por cluster: {PROCESSED_DIR / "rules_por_cluster.pkl"} ({len(rules_por_cluster)} clusters)')

# Salvar df com cluster labels
df.to_pickle(PROCESSED_DIR / 'df_com_clusters.pkl')
print(f'[OK] DataFrame com clusters: {PROCESSED_DIR / "df_com_clusters.pkl"}')


SALVAMENTO DE RESULTADOS
[OK] Regras completas: ..\outputs\dados\regras_completas.csv (7923 regras)


[OK] Itemsets frequentes: ..\outputs\dados\itemsets_frequentes.csv (2467 itemsets)


[OK] Regras por periodo: ..\data\processed\rules_por_periodo.pkl (4 periodos)
[OK] Regras por cluster: ..\data\processed\rules_por_cluster.pkl (2 clusters)
[OK] DataFrame com clusters: ..\data\processed\df_com_clusters.pkl


## Visualizacoes Finais

In [11]:
print('\n' + '='*60)
print('VISUALIZACOES FINAIS')
print('='*60)

# 1. Distribuicao de suporte dos itemsets
fig, ax = plt.subplots(figsize=(10, 6))
ax.hist(itemsets['support'], bins=40, color='#3498db', edgecolor='white', alpha=0.8)
ax.set_xlabel('Suporte')
ax.set_ylabel('Numero de Itemsets')
ax.set_title(f'Distribuicao de Suporte dos Itemsets (n={len(itemsets)})')
ax.axvline(x=min_support, color='red', linestyle='--', label=f'min_support={min_support}')
ax.legend()
plt.tight_layout()
plt.savefig(FIGURAS_DIR / 'distribuicao_suporte_itemsets.png', dpi=300, bbox_inches='tight')
plt.close()
print(f'[OK] distribuicao_suporte_itemsets.png')

# 2. Top 15 regras por lift
if len(rules) > 0:
    fig, ax = plt.subplots(figsize=(12, 8))
    top15 = rules.head(15)
    labels = [f'{r["antecedents_str"][:40]}\n=> {r["consequents_str"][:40]}' for _, r in top15.iterrows()]
    colors = ['#e74c3c' if 'Fatal' in r['consequents_str'] else '#3498db' for _, r in top15.iterrows()]
    bars = ax.barh(range(len(top15)), top15['lift'].values, color=colors, alpha=0.8)
    ax.set_yticks(range(len(top15)))
    ax.set_yticklabels(labels, fontsize=8)
    ax.set_xlabel('Lift')
    ax.set_title('Top 15 Regras de Associacao por Lift')
    ax.invert_yaxis()
    plt.tight_layout()
    plt.savefig(FIGURAS_DIR / 'top15_regras_lift.png', dpi=300, bbox_inches='tight')
    plt.close()
    print(f'[OK] top15_regras_lift.png')

# 3. Scatter suporte x confianca (colorido por lift)
if len(rules) > 0:
    fig, ax = plt.subplots(figsize=(10, 8))
    scatter = ax.scatter(
        rules['support'], rules['confidence'],
        c=rules['lift'], cmap='YlOrRd',
        s=rules['leverage'].abs() * 5000 + 10,
        alpha=0.6, edgecolors='grey', linewidth=0.5
    )
    ax.set_xlabel('Suporte')
    ax.set_ylabel('Confianca')
    ax.set_title(f'Regras de Associacao: Suporte x Confianca (n={len(rules)})')
    cbar = plt.colorbar(scatter, ax=ax)
    cbar.set_label('Lift')
    plt.tight_layout()
    plt.savefig(FIGURAS_DIR / 'scatter_suporte_confianca.png', dpi=300, bbox_inches='tight')
    plt.close()
    print(f'[OK] scatter_suporte_confianca.png')

# 4. Resumo temporal
if len(rules_por_periodo) > 0:
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    
    periodos = sorted(rules_por_periodo.keys())
    n_regras_per = [len(rules_por_periodo[p]) for p in periodos]
    n_grav_per = []
    for p in periodos:
        rp = rules_por_periodo[p]
        n_g = len(rp[rp['consequents'].apply(lambda x: any('Fatal' in str(i) for i in x))])
        n_grav_per.append(n_g)
    
    axes[0].bar(range(len(periodos)), n_regras_per, color='#3498db', alpha=0.8)
    axes[0].set_xticks(range(len(periodos)))
    axes[0].set_xticklabels(periodos, rotation=45)
    axes[0].set_ylabel('Numero de Regras')
    axes[0].set_title('Total de Regras por Periodo')
    
    axes[1].bar(range(len(periodos)), n_grav_per, color='#e74c3c', alpha=0.8)
    axes[1].set_xticks(range(len(periodos)))
    axes[1].set_xticklabels(periodos, rotation=45)
    axes[1].set_ylabel('Regras de Gravidade')
    axes[1].set_title('Regras de Gravidade por Periodo')
    
    plt.tight_layout()
    plt.savefig(FIGURAS_DIR / 'analise_temporal_regras.png', dpi=300, bbox_inches='tight')
    plt.close()
    print(f'[OK] analise_temporal_regras.png')


VISUALIZACOES FINAIS


[OK] distribuicao_suporte_itemsets.png


[OK] top15_regras_lift.png


[OK] scatter_suporte_confianca.png


[OK] analise_temporal_regras.png


## Resumo Final

In [12]:
print('\n' + '='*60)
print('RESUMO DA FASE 4 -- MODELAGEM')
print('='*60)

print(f'\n1. FP-Growth global:')
print(f'   [OK] {len(itemsets)} itemsets frequentes (min_support={min_support})')
print(f'   [OK] Tempo de execucao: {t_fpgrowth:.2f}s')

print(f'\n2. Regras de associacao:')
print(f'   [OK] {len(rules)} regras com lift > 1.0')
if len(rules) > 0:
    print(f'   [OK] Lift max: {rules["lift"].max():.2f}')
    print(f'   [OK] Confianca max: {rules["confidence"].max():.3f}')

print(f'\n3. Regras de gravidade:')
print(f'   [OK] {len(rules_graves)} regras focadas em fatalidade')

print(f'\n4. Grid de parametros:')
print(f'   [OK] {len(df_grid)} combinacoes testadas')

print(f'\n5. Mineracao temporal:')
print(f'   [OK] {len(rules_por_periodo)} periodos analisados')
if len(rules_por_periodo) > 0:
    total_temp = sum(len(r) for r in rules_por_periodo.values())
    print(f'   [OK] {total_temp} regras totais entre periodos')
    if 'regras_estaveis' in dir():
        print(f'   [OK] {len(regras_estaveis)} regras estaveis')

print(f'\n6. Clustering:')
print(f'   [OK] k={k_final} clusters (melhor silhouette)')
print(f'   [OK] {len(rules_por_cluster)} clusters com regras')

print(f'\n{"="*60}')
print('[OK] FASE 4 CONCLUIDA COM SUCESSO')
print(f'{"="*60}')


RESUMO DA FASE 4 -- MODELAGEM

1. FP-Growth global:
   [OK] 2467 itemsets frequentes (min_support=0.05)
   [OK] Tempo de execucao: 0.18s

2. Regras de associacao:
   [OK] 7923 regras com lift > 1.0
   [OK] Lift max: 14.15
   [OK] Confianca max: 1.000

3. Regras de gravidade:
   [OK] 0 regras focadas em fatalidade

4. Grid de parametros:
   [OK] 36 combinacoes testadas

5. Mineracao temporal:
   [OK] 4 periodos analisados
   [OK] 38618 regras totais entre periodos
   [OK] 9021 regras estaveis

6. Clustering:
   [OK] k=2 clusters (melhor silhouette)
   [OK] 2 clusters com regras

[OK] FASE 4 CONCLUIDA COM SUCESSO


## 4.V2 Mineração restrita (contexto → Fatal)

Regras com antecedentes **somente de contexto** e consequente `desfecho_Fatal`. Sem tautologias gravidade↔classificação.

In [ ]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path('..').resolve()))

import pandas as pd
from src.config import DADOS_DIR, MIN_SUPPORT, PROCESSED_DIR, TABELAS_DIR
from src.mining import (
    classificar_estabilidade_temporal,
    comparar_uso_solo,
    minerar_por_ano,
    minerar_regras_fatais,
)
from src.evaluation import traduzir_regras

df_onehot = pd.read_pickle(PROCESSED_DIR / 'transacional.pkl')
df_meta = pd.read_pickle(PROCESSED_DIR / 'transacional_meta.pkl')

itemsets, rules_all, rules_fatal = minerar_regras_fatais(df_onehot, min_support=MIN_SUPPORT)
rules_fatal = traduzir_regras(rules_fatal)

print(f'Itemsets: {len(itemsets):,} | Regras totais: {len(rules_all):,}')
print(f'Regras contexto->Fatal: {len(rules_fatal):,}')
display(rules_fatal.head(15))

rules_por_ano = minerar_por_ano(df_onehot, df_meta)
estabilidade = classificar_estabilidade_temporal(rules_por_ano)
rules_estratos = comparar_uso_solo(df_onehot, df_meta)

rules_fatal.to_csv(DADOS_DIR / 'regras_contexto_fatal.csv', index=False)
if not estabilidade.empty:
    estabilidade.to_csv(TABELAS_DIR / 'regras_estabilidade_temporal.csv', index=False)

print('Estaveis:', (estabilidade['status']=='estavel').sum() if not estabilidade.empty else 0)
for k, v in rules_estratos.items():
    print(f'Estrato {k}: {len(v)} regras')